## DMEPOS-Supplier and Service Data Carpentry
The purpose of this notebook is to clean the DMEPOS-by Supplier and Service dataset for exploration and ML modeling.

Some of the cleaning steps involved:
- Loaded the 2021, 2022, and 2023 CSV files into DataFrames
- Added a Year column to each dataset
- Reordered columns so that Year is the first column
- Combined the three yearly datasets into a single unified DataFrame
- Performed initial descriptive statistics (df.describe()) to understand data distribution
- Checked dataset shape and number of rows/columns (df.shape)
- Counted null values in each column to identify missing data
- Cleaned string columns
- Verified dataset shape and previewed data after cleaning (df.head())


Optional:
- Created a new column DME_Sprsn_Ind to flag suppressed rows in Tot_Suplr_Benes, (less than 10)
- Reordered columns to place DME_Sprsn_Ind before Tot_Suplr_Benes

For information regarding each variable, please refer to the data dictionary downloadable from [DMEPOS - by Supplier and Service](https://data.cms.gov/provider-summary-by-type-of-service/medicare-durable-medical-equipment-devices-supplies/medicare-durable-medical-equipment-devices-supplies-by-supplier-and-service).

End result:

Saved the cleaned dataset to a shared team directory: /dsa/groups/casestudycf25/team02/silver/dmepos_suplr_serv_clean.csv

## Loading the data and quick exploration

In [1]:
# Loading Libraries
import re
import pandas as pd
import csv
from pathlib import Path

Listing all files and folders inside the specified directory

In [2]:
for f in Path("/dsa/groups/casestudycf25/team02/").iterdir():
    print(f.name)

Taxonomy Code List Dec 2023.csv
casestudycf25t02.sqlite.db
OWNRSHP_PGYR2021_2023.csv
bronze
silver
gold
ownership_payment_clean.csv


In [3]:
#Loading the three yearly DMEPOS Supplier–Service datasets (2021–2023)
df21 = pd.read_csv("/dsa/groups/casestudycf25/team02/bronze/mup_dme_ry25_p05_v20_dy21_suphpr.csv",na_values=["NA"])

df22 = pd.read_csv("/dsa/groups/casestudycf25/team02/bronze/mup_dme_ry25_p05_v20_dy22_suphpr.csv",na_values=["NA"])

df23 = pd.read_csv("/dsa/groups/casestudycf25/team02/bronze/mup_dme_ry25_p05_v10_dy23_suphpr.csv",na_values=["NA"])

# Ensure Suplr_Prvdr_RUCA is treated as numeric
for df_ in [df21, df22, df23]:
    df_["Suplr_Prvdr_RUCA"] = pd.to_numeric(df_["Suplr_Prvdr_RUCA"], errors="coerce")

/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3058: DtypeWarning: Columns (2,3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


Taking a quick peak at the dataframes to see if everything is loaded. I can also see the structure of the datasets.

In [4]:
df21.head()

,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,Suplr_Prvdr_State_Abrvtn,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3808,"Wrist hand finger orthosis, rigid without join...",N,69.0,79,82,435.609756,325.202683,257.561707,262.395732
1,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3906,"Wrist hand orthosis, without joints, may inclu...",N,30.0,30,35,448.285714,439.852857,351.134286,315.416286
2,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3908,"Wrist hand orthosis, wrist extension control c...",N,84.0,99,107,108.457944,66.980000,51.914953,46.384393
3,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3913,"Hand finger orthosis, without joints, may incl...",N,16.0,16,16,348.000000,247.410000,197.133125,191.759375
4,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3923,"Hand finger orthosis, without joints, may incl...",N,54.0,59,66,113.030303,88.250000,66.411515,65.491667


In [5]:
df22.head()

,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,Suplr_Prvdr_State_Abrvtn,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3808,"Wrist hand finger orthosis, rigid without join...",N,62.0,62,65,489.230769,342.030000,269.987077,278.113692
1,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3906,"Wrist hand orthosis, without joints, may inclu...",N,31.0,31,32,503.562500,467.656875,363.158125,326.565313
2,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3908,"Wrist hand orthosis, wrist extension control c...",N,61.0,61,67,110.447761,70.400000,53.040746,48.040299
3,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3913,"Hand finger orthosis, without joints, may incl...",N,12.0,12,12,376.000000,260.030000,205.073333,202.370000
4,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3923,"Hand finger orthosis, without joints, may incl...",N,30.0,30,32,115.000000,92.750000,69.985313,69.994062


In [6]:
df23.head()

,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,Suplr_Prvdr_State_Abrvtn,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3808,"Wrist hand finger orthosis, rigid without join...",N,38.0,39,39,540.000000,371.79,289.320256,300.242821
1,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3906,"Wrist hand orthosis, without joints, may inclu...",N,27.0,27,28,555.178571,528.91,414.603214,361.113214
2,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3908,"Wrist hand orthosis, wrist extension control c...",N,57.0,57,63,112.619048,76.52,59.047619,53.890794
3,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3923,"Hand finger orthosis, without joints, may incl...",N,24.0,24,27,115.000000,100.82,78.927778,79.388148
4,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3925,"Finger orthosis, proximal interphalangeal (pip...",N,13.0,13,13,99.769231,68.56,42.296154,34.993077


In [7]:
# Looking at the column names....

df23.columns.tolist()

['Suplr_NPI',
 'Suplr_Prvdr_Last_Name_Org',
 'Suplr_Prvdr_First_Name',
 'Suplr_Prvdr_MI',
 'Suplr_Prvdr_Crdntls',
 'Suplr_Prvdr_Ent_Cd',
 'Suplr_Prvdr_St1',
 'Suplr_Prvdr_St2',
 'Suplr_Prvdr_City',
 'Suplr_Prvdr_State_Abrvtn',
 'Suplr_Prvdr_State_FIPS',
 'Suplr_Prvdr_Zip5',
 'Suplr_Prvdr_RUCA_Cat',
 'Suplr_Prvdr_RUCA',
 'Suplr_Prvdr_RUCA_Desc',
 'Suplr_Prvdr_Cntry',
 'Suplr_Prvdr_Spclty_Cd',
 'Suplr_Prvdr_Spclty_Desc',
 'Suplr_Prvdr_Spclty_Srce',
 'RBCS_Lvl',
 'RBCS_Id',
 'RBCS_Desc',
 'HCPCS_Cd',
 'HCPCS_Desc',
 'Suplr_Rentl_Ind',
 'Tot_Suplr_Benes',
 'Tot_Suplr_Clms',
 'Tot_Suplr_Srvcs',
 'Avg_Suplr_Sbmtd_Chrg',
 'Avg_Suplr_Mdcr_Alowd_Amt',
 'Avg_Suplr_Mdcr_Pymt_Amt',
 'Avg_Suplr_Mdcr_Stdzd_Amt']

Verify the number of records in each file.

In [8]:
len21 = len(df21)
len22 = len(df22)
len23 = len(df23)
print(f'{len21}, {len22}, {len23}')

508052, 482638, 463784


In [9]:
#Checking if columns match..
df21.columns == df23.columns 

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True])

In [10]:
df22.columns == df23.columns 

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True])

## Carpentry

The below code adds a Year column to each yearly dataset, reorders columns so that Year comes first, and then combines all three years into a single DataFrame. The result is one dataset with a consistent structure across 2021, 2022, and 2023, that we can use for analysis.

In [11]:
# Adding a year column
df21["Year"] = 2021
df22["Year"] = 2022
df23["Year"] = 2023

# Reordering columns
df21 = df21[["Year"] + [c for c in df21.columns if c != "Year"]]
df22 = df22[["Year"] + [c for c in df22.columns if c != "Year"]]
df23 = df23[["Year"] + [c for c in df23.columns if c != "Year"]]

# Combining all three datasets
df = pd.concat([df21, df22, df23], ignore_index=True)

In [12]:
# Checking...
df.head()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3808,"Wrist hand finger orthosis, rigid without join...",N,69.0,79,82,435.609756,325.202683,257.561707,262.395732
1,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3906,"Wrist hand orthosis, without joints, may inclu...",N,30.0,30,35,448.285714,439.852857,351.134286,315.416286
2,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3908,"Wrist hand orthosis, wrist extension control c...",N,84.0,99,107,108.457944,66.980000,51.914953,46.384393
3,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3913,"Hand finger orthosis, without joints, may incl...",N,16.0,16,16,348.000000,247.410000,197.133125,191.759375
4,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3923,"Hand finger orthosis, without joints, may incl...",N,54.0,59,66,113.030303,88.250000,66.411515,65.491667


In [13]:
# Checking...
df.tail()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
1454469,2023,1992989909,Joseph P Gabryszewski,NaN,NaN,NaN,O,3 Kirchner Ave,NaN,Hyde Park,...,A5513,"For diabetics only, multiple density insert, c...",N,29.0,29,172,56.62,51.47,40.353372,40.350000
1454470,2023,1992999106,"Northwest Indiana Eye Associates, Pc",NaN,NaN,NaN,O,297 W. Franciscan Dr.,Suite 101,Crown Point,...,V2020,"Frames, purchases",N,72.0,89,89,70.00,70.00,53.646742,61.208989
1454471,2023,1992999106,"Northwest Indiana Eye Associates, Pc",NaN,NaN,NaN,O,297 W. Franciscan Dr.,Suite 101,Crown Point,...,V2103,"Spherocylinder, single vision, plano to plus o...",N,24.0,29,58,45.00,44.71,35.050000,33.850000
1454472,2023,1992999106,"Northwest Indiana Eye Associates, Pc",NaN,NaN,NaN,O,297 W. Franciscan Dr.,Suite 101,Crown Point,...,V2203,"Spherocylinder, bifocal, plano to plus or minu...",N,20.0,28,54,65.00,61.60,48.290000,51.480000
1454473,2023,1992999106,"Northwest Indiana Eye Associates, Pc",NaN,NaN,NaN,O,297 W. Franciscan Dr.,Suite 101,Crown Point,...,V2303,"Spherocylinder, trifocal, plano to plus or min...",N,28.0,30,60,70.00,70.00,46.640667,55.972333


In [14]:
# Looking into descriptive stats
df.describe()

,Year,Suplr_NPI,Suplr_Prvdr_State_FIPS,Suplr_Prvdr_Zip5,Suplr_Prvdr_RUCA,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
count,1.454474e+06,1.454474e+06,1.454474e+06,1.454474e+06,1.454380e+06,864704.000000,1.454474e+06,1.454474e+06,1.454474e+06,1.454474e+06,1.454474e+06,1.454474e+06
mean,2.021970e+03,1.502034e+09,2.856412e+01,4.914111e+04,2.134648e+00,97.770228,1.647799e+02,4.576480e+03,2.056878e+02,1.010626e+02,7.798224e+01,7.767823e+01
std,8.168502e-01,2.883286e+08,1.550741e+01,2.749516e+04,2.640066e+00,620.189075,1.892079e+03,4.952015e+05,9.028967e+02,4.198971e+02,3.283998e+02,3.193373e+02
min,2.021000e+03,1.003000e+09,1.000000e+00,6.050000e+02,1.000000e+00,11.000000,1.100000e+01,1.100000e+01,3.348654e-04,3.348654e-04,0.000000e+00,0.000000e+00
25%,2.021000e+03,1.255406e+09,1.700000e+01,2.836500e+04,1.000000e+00,16.000000,1.600000e+01,2.700000e+01,1.321108e+01,2.799809e+00,2.087560e+00,2.183343e+00
50%,2.022000e+03,1.508413e+09,2.800000e+01,4.656300e+04,1.000000e+00,28.000000,2.900000e+01,9.600000e+01,3.659725e+01,1.600000e+01,1.261647e+01,1.254000e+01
75%,2.023000e+03,1.750331e+09,4.200000e+01,7.274500e+04,2.000000e+00,65.000000,7.900000e+01,6.960000e+02,1.150000e+02,5.661000e+01,4.196140e+01,4.371000e+01
max,2.023000e+03,1.992999e+09,7.800000e+01,9.990100e+04,9.900000e+01,112797.000000,8.590970e+05,2.967321e+08,7.761210e+04,2.773326e+04,2.174288e+04,2.149642e+04


In [15]:
# Data types
df.dtypes

Year                           int64
Suplr_NPI                      int64
Suplr_Prvdr_Last_Name_Org     object
Suplr_Prvdr_First_Name        object
Suplr_Prvdr_MI                object
Suplr_Prvdr_Crdntls           object
Suplr_Prvdr_Ent_Cd            object
Suplr_Prvdr_St1               object
Suplr_Prvdr_St2               object
Suplr_Prvdr_City              object
Suplr_Prvdr_State_Abrvtn      object
Suplr_Prvdr_State_FIPS         int64
Suplr_Prvdr_Zip5               int64
Suplr_Prvdr_RUCA_Cat          object
Suplr_Prvdr_RUCA             float64
Suplr_Prvdr_RUCA_Desc         object
Suplr_Prvdr_Cntry             object
Suplr_Prvdr_Spclty_Cd         object
Suplr_Prvdr_Spclty_Desc       object
Suplr_Prvdr_Spclty_Srce       object
RBCS_Lvl                      object
RBCS_Id                       object
RBCS_Desc                     object
HCPCS_Cd                      object
HCPCS_Desc                    object
Suplr_Rentl_Ind               object
Tot_Suplr_Benes              float64
T

In [16]:
# Shape
df.shape

(1454474, 33)

In [17]:
# Check columns which have missing data
df_nulltots = df.isnull().sum()
df_nulltots

# Most of the nulls are with Tot_Suplr_Benes.

# Like the by supplier dataset, nulls in Suplr_Prvdr_Crdntls will be replaced by ""

Year                               0
Suplr_NPI                          0
Suplr_Prvdr_Last_Name_Org          0
Suplr_Prvdr_First_Name       1450500
Suplr_Prvdr_MI               1451490
Suplr_Prvdr_Crdntls          1451329
Suplr_Prvdr_Ent_Cd                 0
Suplr_Prvdr_St1                    0
Suplr_Prvdr_St2              1237311
Suplr_Prvdr_City                   0
Suplr_Prvdr_State_Abrvtn           0
Suplr_Prvdr_State_FIPS             0
Suplr_Prvdr_Zip5                   0
Suplr_Prvdr_RUCA_Cat              94
Suplr_Prvdr_RUCA                  94
Suplr_Prvdr_RUCA_Desc             94
Suplr_Prvdr_Cntry                  0
Suplr_Prvdr_Spclty_Cd              0
Suplr_Prvdr_Spclty_Desc            0
Suplr_Prvdr_Spclty_Srce            0
RBCS_Lvl                           0
RBCS_Id                            0
RBCS_Desc                          0
HCPCS_Cd                           0
HCPCS_Desc                         0
Suplr_Rentl_Ind                    0
Tot_Suplr_Benes               589770
T

In [18]:
# Normalize credentials to lowercase and remove punctuation
df['Suplr_Prvdr_Crdntls'] = df['Suplr_Prvdr_Crdntls'].fillna('')
df['Suplr_Prvdr_Crdntls'] = df['Suplr_Prvdr_Crdntls'].apply(lambda x: re.sub(r'[^a-zA-Z]','',x).lower())
display(df['Suplr_Prvdr_Crdntls'].unique())
print(df['Suplr_Prvdr_Crdntls'].nunique())

array(['', 'dpm', 'od', 'dds', 'bco', 'cped', 'md', 'certifiedpedorthist',
       'prosthesisfitter', 'cmf', 'pharmacist', 'ddsms', 'mdmba',
       'otrlcht', 'dpmaacfasabpm', 'do', 'rph', 'ddspc', 'mdpa', 'dmdms',
       'mdphd', 'cfm', 'otrcht', 'ot', 'ptcltlana', 'cpboco', 'dmd',
       'bocp', 'ldo', 'dph', 'phd', 'rnmncfm', 'dpminc', 'rncpbocpolpo',
       'cplp', 'pt', 'lcped', 'cplpo', 'rrt', 'mscp', 'odpc', 'optician',
       'cpo', 'dpmpc', 'licensedprosthetist', 'podiatry', 'dpmpa',
       'otrlhtcpamllcc', 'cwsats', 'licensedoptician', 'mastectomyfitter',
       'rfm', 'cof', 'motrlcht', 'aboc', 'cplpcfocfts', 'fnpc', 'dc'],
      dtype=object)

58


Use the 2010 Rural-Urban Commuting Area Codes and ZIP codes table from the US Department of Agriculture to build a mapping of zip codes to rural-urban commuting area (RUCA) codes for imputing null RUCA values.

In [19]:
# Build mapping of zip codes to RUCA
# Use 2010 data to keep consistent with methodology per data dictionary
zip_map = {}

with open('/dsa/groups/casestudycf25/team02/bronze/RUCA2010zipcode.csv', mode='r', newline='') as file:
    csv_reader = csv.DictReader(file)
    # print(csv_reader.fieldnames)
    # row = next(csv_reader)
    # print(re.sub(r"'","",row[csv_reader.fieldnames[0]]))
    for row in csv_reader:
        # write RUCA2 code (value) into zip_map with ZIP_CODE as key
        zip_map[re.sub(r"'","",row[csv_reader.fieldnames[0]])] = float(row['RUCA2']) # Access data using column nam

In [20]:
# Build mapping of RUCA to urban-rural designation and description per https://www.ers.usda.gov/data-products/rural-urban-commuting-area-codes
code_map = {1.:['Urban','Metropolitan area core: primary flow within an urbanized area of 50,000 and greater'],
            1.1:['Urban','Secondary flow 30% to <50% to a larger urbanized area of 50,000 and greater'],
            2.:['Urban','Metropolitan area high commuting: primary flow 30% or more to a urbanized area of 50,000 and greater'],
            2.1:['Urban','Secondary flow 30% to <50% to a larger urbanized area of 50,000 and greater'],
            3.:['Urban','Metropolitan area low commuting: primary flow 10% to <30% to a urbanized area of 50,000 and greater'],
            4.:['Urban','Micropolitan area core: primary flow within an urban cluster of 10,000 to 49,999'],
            4.1:['Urban','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            5.:['Urban','Micropolitan high commuting: primary flow 30% or more to a urban cluster of 10,000 to 49,999'],
            5.1:['Urban','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            6.:['Urban','Micropolitan low commuting: primary flow 10% to <30% to a urban cluster of 10,000 to 49,999'],
            7.:['Urban','Small town core: primary flow within an urban cluster of 2,500 to 9,999'],
            7.1:['Urban','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            7.2:['Urban','Secondary flow 30% to <50% to a urban cluster of 10,000 to 49,999'],
            8.:['Urban','Small town high commuting: primary flow 30% or more to a urban cluster of 2,500 to 9,999'],
            8.1:['Urban','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            8.2:['Urban','Secondary flow 30% to <50% to a urban cluster of 10,000 to 49,999'],
            9.:['Urban','Small town low commuting: primary flow 10% to <30% to a urban cluster of 2,500 to 9,999'],
            10.:['Rural','Rural areas: primary flow to a tract outside a urbanized area of 50,000 and greater or UC'],
            10.1:['Rural','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            10.2:['Rural','Secondary flow 30% to <50% to a urban cluster of 10,000 to 49,999'],
            10.3:['Rural','Secondary flow 30% to <50% to a urban cluster of 2,500 to 9,999'],
            99.:['Unknown','Unknown']}
            

In [21]:
missing_zips = df[df.Suplr_Prvdr_RUCA_Cat.isnull()]['Suplr_Prvdr_Zip5'].unique()
missing_zips

array([ 6748, 92188, 99686, 85144,  6825,  6824])

In [22]:
# Some zip codes are missing from RUCA2010zipcode.csv
for missing_zip in missing_zips:
    try:
        print(f"{missing_zip} maps to {zip_map[missing_zip]}")
    except:
        zip_map[missing_zip] = 99. # map zip code to 99 for unknown

In [23]:
# Link missing RUCA by zip code from table at https://www.ers.usda.gov/data-products/rural-urban-commuting-area-codes
temp  = df
df.Suplr_Prvdr_RUCA = temp.Suplr_Prvdr_RUCA.fillna(temp.Suplr_Prvdr_Zip5.map(zip_map))

In [24]:
# Map in missing RUCA_Cat and Desc
target_columns = ['Suplr_Prvdr_RUCA_Cat', 'Suplr_Prvdr_RUCA_Desc']

for i, col in enumerate(target_columns):
    df[col] = df.apply(lambda row: code_map[row['Suplr_Prvdr_RUCA']][i] if pd.isna(row[col]) else row[col], axis=1)

Impute missing specialty codes:

The mapping of specialty descriptions to codes from the CMS Medicare Provider and Supplier Taxonomy Crosswalk was done manually based on judgment, in line with the approach used for the referral provider dataset. Descriptions that did not have a clear or direct match, or were too ambiguous, were left blank.

Link: https://data.cms.gov/provider-characteristics/medicare-provider-supplier-enrollment/medicare-provider-and-supplier-taxonomy-crosswalk


In [25]:
specialty_map = {'Anesthesiology': '5','Clinic/Center': '70','Clinical Neuropsychologist': '86','Colon & Rectal Surgery': '28','Counselor': 'E2','Dentist': 'C5','Emergency Medicine': '93','Family Medicine': '8',
                 'General Acute Care Hospital': 'A0','General Practice': '1','Health Maintenance Organization': '58','Home Health': 'A4','Integrative Medicine': '',
                 'Internal Medicine': '11','Legal Medicine': '','Licensed Practical Nurse': '89','Local Education Agency (LEA)': '','Midwife': '42',
                 'Military Health Care Provider': 'A0','Naturopath': '','Neurological Surgery': '14','Neuromusculoskeletal Medicine, Sports Medicine': '23','Obstetrics & Gynecology': '16',
                 'Oral & Maxillofacial Surgery': '19','Orthopaedic Surgery': '20','Otolaryngology': '4','Pain Medicine': '72','Pediatrics': '37','Pedorthist': 'B2',
                 'Personal Emergency Response Attendant': '','Pharmacist': 'A5','Pharmacy Technician': 'A5','Physical Medicine & Rehabilitation': '25',
                 'Plastic Surgery': '24','Prosthetist': '56','Psychiatry & Neurology': '26','Psychoanalyst': '62','Psychologist': '62',
                 'Registered Nurse': '89','Respite Care': '','Sleep Specialist, PhD': 'C0','Social Worker': '80','Specialist': '',
                 'Specialist/Technologist, Other': '75','Student in an Organized Health Care Education/Training Program': '',
                 'Surgery': '2','Thoracic Surgery (Cardiothoracic Vascular Surgery)': '33'}

In [26]:
df.Suplr_Prvdr_Spclty_Cd = df.Suplr_Prvdr_Spclty_Cd.fillna(df.Suplr_Prvdr_Spclty_Desc.map(specialty_map))

In [27]:
# transform the data
df['Suplr_Prvdr_Spclty_Desc'] = df['Suplr_Prvdr_Spclty_Desc'].apply(lambda x: re.sub(r'[^a-z0-9_]', '', str(x).lower().replace(' ', '_')))

In [28]:
# Replace Tot_Suplr_Benes nulls with 5
df["Tot_Suplr_Benes"] = df["Tot_Suplr_Benes"].fillna(5)

In [29]:
df.isnull().sum()

Year                               0
Suplr_NPI                          0
Suplr_Prvdr_Last_Name_Org          0
Suplr_Prvdr_First_Name       1450500
Suplr_Prvdr_MI               1451490
Suplr_Prvdr_Crdntls                0
Suplr_Prvdr_Ent_Cd                 0
Suplr_Prvdr_St1                    0
Suplr_Prvdr_St2              1237311
Suplr_Prvdr_City                   0
Suplr_Prvdr_State_Abrvtn           0
Suplr_Prvdr_State_FIPS             0
Suplr_Prvdr_Zip5                   0
Suplr_Prvdr_RUCA_Cat               0
Suplr_Prvdr_RUCA                   0
Suplr_Prvdr_RUCA_Desc              0
Suplr_Prvdr_Cntry                  0
Suplr_Prvdr_Spclty_Cd              0
Suplr_Prvdr_Spclty_Desc            0
Suplr_Prvdr_Spclty_Srce            0
RBCS_Lvl                           0
RBCS_Id                            0
RBCS_Desc                          0
HCPCS_Cd                           0
HCPCS_Desc                         0
Suplr_Rentl_Ind                    0
Tot_Suplr_Benes                    0
T

In [30]:
# Checking ..
df.head()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3808,"Wrist hand finger orthosis, rigid without join...",N,69.0,79,82,435.609756,325.202683,257.561707,262.395732
1,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3906,"Wrist hand orthosis, without joints, may inclu...",N,30.0,30,35,448.285714,439.852857,351.134286,315.416286
2,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3908,"Wrist hand orthosis, wrist extension control c...",N,84.0,99,107,108.457944,66.980000,51.914953,46.384393
3,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3913,"Hand finger orthosis, without joints, may incl...",N,16.0,16,16,348.000000,247.410000,197.133125,191.759375
4,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3923,"Hand finger orthosis, without joints, may incl...",N,54.0,59,66,113.030303,88.250000,66.411515,65.491667


In [31]:
# Converting column names to snake case

df.columns = (df.columns.str.lower())

In [32]:
# Looking at column names
df.columns.tolist()

['year',
 'suplr_npi',
 'suplr_prvdr_last_name_org',
 'suplr_prvdr_first_name',
 'suplr_prvdr_mi',
 'suplr_prvdr_crdntls',
 'suplr_prvdr_ent_cd',
 'suplr_prvdr_st1',
 'suplr_prvdr_st2',
 'suplr_prvdr_city',
 'suplr_prvdr_state_abrvtn',
 'suplr_prvdr_state_fips',
 'suplr_prvdr_zip5',
 'suplr_prvdr_ruca_cat',
 'suplr_prvdr_ruca',
 'suplr_prvdr_ruca_desc',
 'suplr_prvdr_cntry',
 'suplr_prvdr_spclty_cd',
 'suplr_prvdr_spclty_desc',
 'suplr_prvdr_spclty_srce',
 'rbcs_lvl',
 'rbcs_id',
 'rbcs_desc',
 'hcpcs_cd',
 'hcpcs_desc',
 'suplr_rentl_ind',
 'tot_suplr_benes',
 'tot_suplr_clms',
 'tot_suplr_srvcs',
 'avg_suplr_sbmtd_chrg',
 'avg_suplr_mdcr_alowd_amt',
 'avg_suplr_mdcr_pymt_amt',
 'avg_suplr_mdcr_stdzd_amt']

In [33]:
# Checking..
df.head(10)

,year,suplr_npi,suplr_prvdr_last_name_org,suplr_prvdr_first_name,suplr_prvdr_mi,suplr_prvdr_crdntls,suplr_prvdr_ent_cd,suplr_prvdr_st1,suplr_prvdr_st2,suplr_prvdr_city,...,hcpcs_cd,hcpcs_desc,suplr_rentl_ind,tot_suplr_benes,tot_suplr_clms,tot_suplr_srvcs,avg_suplr_sbmtd_chrg,avg_suplr_mdcr_alowd_amt,avg_suplr_mdcr_pymt_amt,avg_suplr_mdcr_stdzd_amt
0,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3808,"Wrist hand finger orthosis, rigid without join...",N,69.0,79,82,435.609756,325.202683,257.561707,262.395732
1,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3906,"Wrist hand orthosis, without joints, may inclu...",N,30.0,30,35,448.285714,439.852857,351.134286,315.416286
2,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3908,"Wrist hand orthosis, wrist extension control c...",N,84.0,99,107,108.457944,66.980000,51.914953,46.384393
3,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3913,"Hand finger orthosis, without joints, may incl...",N,16.0,16,16,348.000000,247.410000,197.133125,191.759375
4,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3923,"Hand finger orthosis, without joints, may incl...",N,54.0,59,66,113.030303,88.250000,66.411515,65.491667
5,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3933,"Finger orthosis, without joints, may include s...",N,23.0,26,28,162.857143,162.857143,121.714286,143.888214
6,2021,1003002254,Walgreen Co.,NaN,NaN,,O,5104 Bobby Hicks Hwy,NaN,Gray,...,A4253,Blood glucose test or reagent strips for home ...,N,40.0,81,267,72.825543,8.320000,5.558876,5.466255
7,2021,1003002254,Walgreen Co.,NaN,NaN,,O,5104 Bobby Hicks Hwy,NaN,Gray,...,A4259,"Lancets, per box of 100",N,22.0,36,52,16.214231,1.420000,1.071731,1.054615
8,2021,1003002254,Walgreen Co.,NaN,NaN,,O,5104 Bobby Hicks Hwy,NaN,Gray,...,J7507,"Tacrolimus, immediate release, oral, 1 mg",N,5.0,11,1388,2.788321,0.557875,0.416794,0.408458
9,2021,1003002254,Walgreen Co.,NaN,NaN,,O,5104 Bobby Hicks Hwy,NaN,Gray,...,J7620,"Albuterol, up to 2.5 mg and ipratropium bromid...",N,5.0,12,1800,2.111822,0.118133,0.094500,0.092617


In [34]:
#Saving the cleaned csv....
df.to_csv("/dsa/groups/casestudycf25/team02/silver/dmepos_suplr_serv_clean.csv", index=False)

In [35]:
#Verification
df = pd.read_csv("/dsa/groups/casestudycf25/team02/silver/dmepos_suplr_serv_clean.csv")

In [36]:
check_len = len21 + len22 + len23
df_len = len(df)
print(f'{check_len} versus {df_len}')

1454474 versus 1454474


## Additional changes that can be implemented....